# Experiment brief: voucher cho khách đang hoạt động, và readout chạy thử trước khi chạy thật

Ở Basic 02, bạn đã trả lời trưởng nhóm CRM rằng A/B test email chào mừng cho **khách mới** không khả thi (mỗi
tuần chỉ khoảng 32 khách mới). Trưởng nhóm CRM đổi đề xuất:

> "Vậy thử trên **khách đang hoạt động** (có mua trong 90 ngày qua): gửi một email kèm **voucher giảm 10%** dùng
> trong 30 ngày, đầu tháng 9 trước mùa Giáng sinh. Viết giúp mình experiment brief để trình Giám đốc kinh doanh,
> và cho mình xem trước readout sẽ trông thế nào."

Việc của bạn: dựng các khối tính toán của brief trên dữ liệu thật (đối tượng, baseline, chia nhóm, sample size,
readout), **chạy thử readout bằng A/A test** trên dữ liệu lịch sử (chia ngẫu nhiên nhưng không ai nhận voucher:
mọi khác biệt là nhiễu), rồi viết `BRIEF.md`.

Ngày bắt đầu giả định của thí nghiệm là **2011-09-01** (thí nghiệm là kịch bản giả định; dữ liệu là thật: bộ
`online-retail-ii`, hai sheet, định nghĩa dòng hợp lệ chung của track).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/11-stats-experimentation/exercises/advanced/01-experiment-brief -q`

In [ ]:
import hashlib
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


def customer_invoices(raw: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi hoá đơn hợp lệ có mã khách: customer_id, invoice, invoice_date, revenue. (Đã viết sẵn, như Basic 02.)"""
    valid = raw[~raw["invoice"].astype(str).str.startswith("C") & (raw["quantity"] > 0) & (raw["price"] > 0)
                & raw["customer_id"].notna()]
    valid = valid.assign(invoice=valid["invoice"].astype(str), revenue=valid["quantity"] * valid["price"])
    return valid.groupby(["customer_id", "invoice"], as_index=False).agg(
        invoice_date=("invoice_date", "min"), revenue=("revenue", "sum"))


def proportion_sample_size(p0: float, mde_abs: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Số người mỗi nhóm cho metric tỉ lệ, hai phía (công thức của Basic 02). (Đã viết sẵn.)"""
    p1 = p0 + mde_abs
    p_bar = (p0 + p1) / 2
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    num = z_a * math.sqrt(2 * p_bar * (1 - p_bar)) + z_b * math.sqrt(p0 * (1 - p0) + p1 * (1 - p1))
    return math.ceil(num**2 / mde_abs**2)

## Phần A. Đối tượng và metric

`eligible_customers(invoices, as_of, lookback_days=90) -> Index`: khách **đang hoạt động** tại ngày `as_of`: có
ít nhất một hoá đơn với `invoice_date` trong `[as_of - lookback_days ngày, as_of)` (nửa mở: hoá đơn đúng lúc
`as_of` thuộc giai đoạn thí nghiệm, không thuộc lịch sử). Trả `pd.Index` các `customer_id` duy nhất, **sắp tăng
dần**, tên `customer_id`.

`outcome_window(invoices, customers, start, days=30) -> DataFrame`: kết quả của từng khách trong `customers`
trong cửa sổ `[start, start + days ngày)`: index là `customers` (giữ nguyên thứ tự), cột `purchased` (bool: có ít
nhất một hoá đơn trong cửa sổ) và `revenue` (float: tổng doanh thu trong cửa sổ, **0** nếu không mua). Khách không
có trong `customers` không xuất hiện; khách không mua vẫn phải có dòng (bỏ họ là lỗi kinh điển: metric chỉ còn
tính trên người mua).

In [ ]:
def eligible_customers(invoices: pd.DataFrame, as_of, lookback_days: int = 90) -> pd.Index:
    """customer_id có hoá đơn trong [as_of - lookback_days, as_of), sắp tăng dần, tên "customer_id"."""
    # TODO
    raise NotImplementedError


def outcome_window(invoices: pd.DataFrame, customers, start, days: int = 30) -> pd.DataFrame:
    """Mỗi khách trong customers: purchased (bool), revenue (float, 0 nếu không mua) trong [start, start + days)."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: nạp Online Retail II (lần đầu ~8 giây, có cache parquet trong thư mục tạm)
import sys
import os
import tempfile
import uuid
from pathlib import Path

import duckdb
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure  # noqa: E402

SHEETS = ("Year 2009-2010", "Year 2010-2011")  # sheet đầu chứa lại 01-09/12/2010 của sheet sau: bỏ phần trùng
CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache parquet dùng chung với module 03, không ghi vào data/raw


def _sql_path(path: Path) -> str:
    return str(path).replace("'", "''")


def _sheet_sql(xlsx: Path, sheet: str) -> str:
    return f"""
        SELECT Invoice AS invoice, StockCode AS stock_code, Description AS description,
               CAST(CAST(Quantity AS DOUBLE) AS INTEGER) AS quantity,
               TIMESTAMP '1899-12-30' + to_seconds(CAST(round(CAST(InvoiceDate AS DOUBLE) * 86400) AS BIGINT))
                   AS invoice_date,
               CAST(Price AS DOUBLE) AS price,
               CAST(CAST("Customer ID" AS DOUBLE) AS INTEGER) AS customer_id,
               Country AS country
        FROM read_xlsx('{_sql_path(xlsx)}', sheet = '{sheet}', all_varchar = true)"""


def _excel_extension_missing(exc: duckdb.Error) -> bool:
    """Chỉ lỗi cài/nạp extension excel (máy offline...). Lỗi SQL hay lỗi dữ liệu thì phải nổi lên."""
    kinds = (duckdb.CatalogException, duckdb.IOException, duckdb.HTTPException)
    return isinstance(exc, kinds) and "extension" in str(exc).lower()


def load_online_retail(xlsx: Path) -> pd.DataFrame:
    """Hai sheet Online Retail II gộp (chưa lọc), cột snake_case: invoice, stock_code, description, quantity,
    invoice_date, price, customer_id, country.

    Lần đầu đọc xlsx bằng extension excel của DuckDB (~8 giây; máy không cài được extension thì pandas ~60-90 giây)
    rồi ghi cache parquet vào thư mục tạm của hệ điều hành, khoá theo mtime và kích thước tệp; lần sau ~1 giây.
    """
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_ii_{stat.st_mtime_ns}_{stat.st_size}.parquet"
    con = duckdb.connect()
    if cache.exists():
        try:
            return con.execute(f"SELECT * FROM read_parquet('{_sql_path(cache)}')").df()
        except duckdb.IOException:  # cache hỏng: đọc lại từ xlsx và ghi đè
            pass
    try:
        con.execute(
            f"CREATE TABLE online_retail AS SELECT * FROM ({_sheet_sql(xlsx, SHEETS[0])}) "
            f"WHERE invoice_date < TIMESTAMP '2010-12-01' UNION ALL {_sheet_sql(xlsx, SHEETS[1])}"
        )
    except duckdb.Error as exc:
        if not _excel_extension_missing(exc):
            raise
        old, new = (pd.read_excel(xlsx, sheet_name=s) for s in SHEETS)
        raw = pd.concat([old[old.InvoiceDate < "2010-12-01"], new], ignore_index=True)
        frame = pd.DataFrame({
            "invoice": raw["Invoice"].astype(str), "stock_code": raw["StockCode"].astype(str),
            "description": raw["Description"], "quantity": raw["Quantity"].astype("int32"),
            "invoice_date": raw["InvoiceDate"], "price": raw["Price"].astype(float),
            "customer_id": raw["Customer ID"].astype("Int32"), "country": raw["Country"],
        })
        con.register("frame", frame)
        con.execute("CREATE TABLE online_retail AS SELECT * FROM frame")
    tmp = CACHE_DIR / f".{cache.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp"
    try:  # ghi atomic: tệp tạm tên riêng rồi rename, an toàn khi nhiều tiến trình cùng ghi
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        con.execute(f"COPY online_retail TO '{_sql_path(tmp)}' (FORMAT parquet)")
        os.replace(tmp, cache)
    except (OSError, duckdb.Error):  # không ghi được cache thì thôi, lần sau đọc lại xlsx
        tmp.unlink(missing_ok=True)
    return con.execute("SELECT * FROM online_retail").df()


raw = load_online_retail(ensure("online-retail-ii") / "online_retail_II.xlsx")
invoices = customer_invoices(raw)
print(f"{len(raw):,} dòng, {len(invoices):,} hoá đơn có mã khách; dữ liệu tới {raw['invoice_date'].max()}")

In [ ]:
# explore: đối tượng và baseline 30 ngày ở vài ngày bắt đầu khác nhau (mùa vụ!)
START = "2011-09-01"
rows = []
for as_of in ["2010-09-01", "2011-03-01", "2011-06-01", "2011-08-01", START]:
    people = eligible_customers(invoices, as_of)
    out = outcome_window(invoices, people, as_of, 30)
    rows.append({"as_of": as_of, "eligible": len(people), "purchase_rate": out["purchased"].mean(),
                 "revenue_mean": out["revenue"].mean(), "revenue_sd": out["revenue"].std(),
                 "revenue_p99": out["revenue"].quantile(0.99), "revenue_max": out["revenue"].max()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

eligible = eligible_customers(invoices, START)
outcome = outcome_window(invoices, eligible, START, 30)  # dữ liệu tháng 9/2011 thật: dùng cho A/A ở phần C

In [ ]:
# explore: baseline cho guardrail "huỷ đơn": tỉ lệ doanh thu bị huỷ của nhóm đối tượng trong 30 ngày
window = (raw["invoice_date"] >= START) & (raw["invoice_date"] < pd.Timestamp(START) + pd.Timedelta(days=30))
mine = raw[window & raw["customer_id"].isin(eligible)]
cancel = mine[mine["invoice"].astype(str).str.startswith("C")]
gross = outcome["revenue"].sum()
cancelled = -(cancel["quantity"] * cancel["price"]).sum()
print(f"doanh thu gộp {gross:,.0f}; giá trị dòng huỷ {cancelled:,.0f} ({cancelled / gross:.1%}); "
      f"{cancel['customer_id'].nunique()} / {len(eligible)} khách có ít nhất một dòng huỷ")

## Phần B. Chia nhóm và sample size

`assign_arm(customer_ids, salt, treatment_share=0.5) -> Series`: chia nhóm **ổn định theo hash** (cùng khách, cùng
`salt` luôn vào cùng nhóm, không phụ thuộc thứ tự hay lần chạy; đổi `salt` cho mỗi thí nghiệm để các thí nghiệm
không chia giống hệt nhau). Quy tắc chính xác (test kiểm từng khách):

- chuỗi `f"{salt}:{int(customer_id)}"`, mã hoá UTF-8, `hashlib.sha256(...).hexdigest()`;
- `bucket = int(hexdigest[:8], 16) % 10_000`;
- nhóm `"treatment"` nếu `bucket < round(treatment_share * 10_000)`, ngược lại `"control"`.

Trả Series index là `customer_ids` (giữ thứ tự), giá trị `"treatment"`/`"control"`, tên `arm`.

`mean_sample_size(sd, mde_abs, alpha=0.05, power=0.8) -> int`: số người mỗi nhóm để phát hiện chênh lệch trung
bình `mde_abs` khi độ lệch chuẩn của metric là `sd` (hai phía, hai nhóm bằng nhau), làm tròn lên:

$$n = \frac{2\,(z_{1-\alpha/2} + z_{1-\beta})^2\,\sigma^2}{\text{MDE}^2}$$

Công thức cho thấy vì sao metric doanh thu "khó": cỡ mẫu tỉ lệ với **phương sai**, mà doanh thu/khách có đuôi rất
dài (vài khách sỉ mua hàng chục nghìn GBP một tháng). Cắt đuôi (cap/winsorize) ở một ngưỡng **đặt trước** là cách
giảm phương sai phổ biến nhất.

In [ ]:
def assign_arm(customer_ids, salt: str, treatment_share: float = 0.5) -> pd.Series:
    """Nhóm "treatment"/"control" theo sha256(f"{salt}:{int(id)}"), ổn định giữa các lần chạy."""
    # TODO
    raise NotImplementedError


def mean_sample_size(sd: float, mde_abs: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Số người mỗi nhóm cho metric trung bình, hai phía, làm tròn lên."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: chia nhóm thật cho thí nghiệm, kiểm SRM ngay từ lúc chia
SALT = "welcome-voucher-2011"
arm = assign_arm(eligible, SALT)
counts = arm.value_counts()
print(counts.to_string())
print("SRM so với 50/50:", stats.chisquare(counts[["treatment", "control"]], [len(arm) / 2] * 2))

# explore: sample size cho hai lựa chọn metric chính
p0 = outcome["purchased"].mean()
per_arm = len(eligible) // 2
print(f"\nbaseline mua trong 30 ngày p0 = {p0:.4f}; mỗi nhóm có khoảng {per_arm} khách")
for mde in (0.03, 0.05, 0.07):
    print(f"  tỉ lệ mua +{mde * 100:.0f} điểm %: cần {proportion_sample_size(p0, mde):,} mỗi nhóm")

cap = outcome["revenue"].quantile(0.99)
sd_raw, sd_cap = outcome["revenue"].std(), outcome["revenue"].clip(upper=cap).std()
mean_rev, mean_cap = outcome["revenue"].mean(), outcome["revenue"].clip(upper=cap).mean()
print(f"\ndoanh thu 30 ngày: TB {mean_rev:,.1f}, sd {sd_raw:,.1f}; "
      f"cắt ở p99 = {cap:,.1f} thì TB {mean_cap:,.1f}, sd {sd_cap:,.1f}")
for rel in (0.10, 0.20, 0.30):  # MDE tương đối so với trung bình của chính metric đó
    print(f"  doanh thu/khách +{rel:.0%}: không cắt {mean_sample_size(sd_raw, rel * mean_rev):,}, "
          f"cắt p99 {mean_sample_size(sd_cap, rel * mean_cap):,} mỗi nhóm")

## Phần C. Readout và A/A test

`readout(outcome, arm, cap=None, alpha=0.05) -> DataFrame`: bảng kết quả của thí nghiệm, index
`["purchased", "revenue"]`, cột `["control", "treatment", "diff", "ci_low", "ci_high", "p"]`:

- `arm` là Series cùng index với `outcome` (giá trị `"treatment"`/`"control"`).
- `purchased`: `control`, `treatment` là tỉ lệ mua; `diff = treatment - control`; khoảng tin cậy Wald với phương
  sai riêng hai nhóm; `p` hai phía của z-test với tỉ lệ gộp (như Intermediate 01).
- `revenue`: nếu `cap` khác `None` thì `clip(upper=cap)` trước; `control`, `treatment` là trung bình;
  khoảng tin cậy và `p` của **Welch t-test** (như Basic 03; khớp `scipy.stats.ttest_ind(..., equal_var=False)`).

`aa_false_positive_rate(outcome, salts, metric="purchased", cap=None, alpha=0.05) -> float`: với mỗi `salt`, chia
lại toàn bộ `outcome.index` bằng `assign_arm(..., salt)`, chạy `readout`, đếm số lần `p < alpha` của dòng
`metric`; trả tỉ lệ. Không ai nhận voucher, nên tỉ lệ này phải **gần alpha**: lớn hơn rõ nghĩa là quy trình phân
tích có lỗi (công thức sai, đơn vị phân tích khác đơn vị chia nhóm, ...).

In [ ]:
def readout(outcome: pd.DataFrame, arm: pd.Series, cap: float | None = None, alpha: float = 0.05) -> pd.DataFrame:
    """Bảng readout hai dòng purchased, revenue: control, treatment, diff, ci_low, ci_high, p."""
    # TODO
    raise NotImplementedError


def aa_false_positive_rate(outcome: pd.DataFrame, salts, metric: str = "purchased", cap: float | None = None,
                           alpha: float = 0.05) -> float:
    """Tỉ lệ A/A test (mỗi salt một lần chia) có p < alpha ở dòng metric."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: readout chạy thử trên dữ liệu tháng 9/2011 thật, chia theo SALT của thí nghiệm (A/A: không ai nhận voucher)
print(readout(outcome, arm).round(4).to_string())
print("\ncắt doanh thu ở p99:")
print(readout(outcome, arm, cap=cap).round(4).to_string())

# explore: 200 lần A/A, phân phối p-value phải gần đều trên [0, 1]
salts = [f"aa-{k}" for k in range(200)]
p_values = [readout(outcome, assign_arm(outcome.index, s)).loc["purchased", "p"] for s in salts]
print("\nA/A false positive (purchased):", aa_false_positive_rate(outcome, salts))
print("A/A false positive (revenue, cắt p99):", aa_false_positive_rate(outcome, salts, metric="revenue", cap=cap))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(p_values, bins=20, range=(0, 1), color="#4c78a8", edgecolor="white")
ax.axhline(len(p_values) / 20, color="#e45756", lw=1)
ax.set_xlabel("p-value của A/A test (purchased)")
ax.set_ylabel("số lần")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("200 lần A/A: p-value trải đều, quy trình readout không tự tạo ra \"hiệu ứng\"", loc="left")
plt.show()

## Viết BRIEF.md

Mở `starter/BRIEF.md` và viết brief cho Giám đốc kinh doanh (400-1 000 từ, đủ các mục có sẵn, xoá mọi dòng TODO).
Brief phải trả lời được trước khi chạy: thí nghiệm để ra quyết định gì, ai vào thí nghiệm, đo gì, cần bao nhiêu
người và bao lâu, khi nào **không** được đọc kết quả, quy tắc ship đặt trước, và readout sẽ trông thế nào (dùng
bảng A/A ở trên làm mẫu, ghi rõ đó là chạy thử).

Câu hỏi để tự kiểm trước khi viết:

1. Với khoảng 960 khách mỗi nhóm, MDE của tỉ lệ mua là bao nhiêu? Metric nào làm metric chính được, metric nào chỉ
   là phụ? Có nên dùng doanh thu không cắt đuôi không?
2. Voucher 10% làm doanh thu trên mỗi đơn giảm: guardrail nào bảo vệ biên lợi nhuận? Chọn metric chính đo theo
   giá niêm yết hay sau voucher, rồi tính doanh thu phải tăng bao nhiêu thì voucher hoà vốn **trên đúng metric đó**
   (hai cách đo cho hai ngưỡng khác nhau). MDE của thí nghiệm nằm trên hay dưới ngưỡng đó, và điều đó nghĩa là gì?
3. Thí nghiệm bắt đầu 1/9: baseline tháng 9 khác tháng 3, 6, 8 thế nào? Điều đó ảnh hưởng gì tới sample size và
   tới việc "dùng kết quả tháng 9 cho cả năm"?
4. Nếu khách nhóm control nghe tin voucher (chia sẻ mã), kết quả lệch về phía nào?